# 07 Base model B

Learns only from normal flights, so it is not pulled towards the rare extreme labels and is more precise for the
bulk of the traffic. It still predicts every flight; the corrector decides how much to trust it.

In [1]:
OOF_MONTHS = None        # None = an out-of-fold prediction for every month of 2025
MAX_PARALLEL_BLOCKS = 6

## Features

In [2]:
FEATURES = [
    # airport, aircraft and runway
    "ADEP_mvt", "AIRCRAFT_TYPE_mvt", "RUNWAY_mvt", "AIRCRAFT_OPERATOR_flt",
    # NM and schedule times
    "mvt_minus_aobt_sec", "movement_minus_scheduled_offblock_sec", "mvt_minus_lobt_sec", "aobt_minus_eobt_sec",
    "lobt_minus_eobt_sec",
    # traffic, weather and day of week
    "deps_before_30min", "deps_before_30min_same_runway", "deps_after_30min_same_runway", "seconds_since_last_movement",
    "dep_median_delay_30min_sec", "weather_temperature_c", "weather_visibility_km",
    "movement_day_of_week_sin", "movement_day_of_week_cos",
    # ADS-B and stand
    "adsb_movement_to_mvt_sec", "adsb_first_observation_to_mvt_sec", "adsb_movement_to_runway_direct_m",
    "stand_reocc_gap_sec", "stand_reocc_other_type", "stand_prev_arr_gap_sec",
]
FEATURES += [
    # departure flow and local hour
    "dep_rate_60min_ratio", "dep_runway_share_30min", "dep_n_runways_60min", "deps_same_runway_10min",
    "local_hour", "dep_max_gap_2h_sec",
    # queue in front of the departure and runway configuration
    "nbr_mvt_minus_aobt_30min_sec", "nbr_mvt_minus_aobt_60min_sec", "nbr_mvt_minus_aobt_rwy_60min_sec",
    "deps_between_offblock_and_mvt", "deps_same_runway_between_offblock_and_mvt", "taxi_queue_at_offblock",
    "gate_retention_at_offblock", "arr_taxiing_at_offblock", "dep_runway_config_60min", "arr_runway_config_60min",
    "aobt_has_seconds",
    # stand and runway geometry
    "taxi_route_distance_m", "stand_to_runway_distance_m", "stand_runway_parallel_m", "stand_runway_perpendicular_m",
    "runway_length_m", "WK_TBL_CAT_flt",
]

## Model

CatBoost regressor, depth 8, learning rate 0.05, up to 20 000 trees with early stopping. Text columns are passed as
categorical features.

In [3]:
from catboost import CatBoostRegressor

MODEL = CatBoostRegressor(iterations=20000, depth=8, learning_rate=0.05, border_count=4095, loss_function="RMSE",
                          cat_features=["ADEP_mvt", "AIRCRAFT_TYPE_mvt", "RUNWAY_mvt", "AIRCRAFT_OPERATOR_flt",
                                        "dep_runway_config_60min", "arr_runway_config_60min", "WK_TBL_CAT_flt"],
                          verbose=False, allow_writing_files=False)

## Data

In [4]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from IPython.display import display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "model/runner.py").is_file())
sys.path.insert(0, str(ROOT / "model"))
import runner
from runner import TARGET, TIME

runner.configure_tracking()
DYNAMIC = {name: (runner.FEATURE_DIR / f"{name}.py").read_text() for name in ["median_taxi_out"]}
needed = list(dict.fromkeys(FEATURES + ["STAND_mvt", "MVT_TIME_UTC_mvt", TARGET, TIME, *runner.METADATA]))
data = pd.read_parquet(ROOT / "data/03_features/train_DEP_merged_sorted.parquet", columns=needed)
data = data.loc[pd.to_datetime(data[TIME], utc=True).dt.year.eq(2025)].reset_index(drop=True)
print(f"{len(data):,} departures in 2025, {len(FEATURES)} features + {list(DYNAMIC)}")

2,085,041 departures in 2025, 47 features + ['median_taxi_out']


## Normal flights

Training rows: taxi time up to one hour and not an artefact (label equal to the delay with a delay above 2000 s).

In [5]:
MAX_TAXI_SEC = 3600            # longer taxi times are left out of training
ARTEFACT_TOLERANCE_SEC = 60
ARTEFACT_MIN_DELAY_SEC = 2000  # below this, label = delay simply means an on-time pushback

delay = (pd.to_datetime(data["MVT_TIME_UTC_mvt"], utc=True) - pd.to_datetime(data[TIME], utc=True)).dt.total_seconds()
artefact = ((data[TARGET] - delay).abs() <= ARTEFACT_TOLERANCE_SEC) & (delay > ARTEFACT_MIN_DELAY_SEC)
fit_mask = ((data[TARGET] <= MAX_TAXI_SEC) & ~artefact).to_numpy()
print(f"Normal flights used for training: {fit_mask.mean():.1%} of 2025")

Normal flights used for training: 99.5% of 2025


## Month blocks

Each month of 2025 is predicted by a model trained on the other eleven months. Early stopping uses the month before
the block (for January the month after it); with that number of trees the model is then refitted on all eleven
months. These out-of-fold predictions are what the corrector learns from.

In [6]:
blocks = runner.month_blocks(OOF_MONTHS)
run, oof, metrics = runner.train_blocks("BaseB_CatBoost", MODEL, data, FEATURES + list(DYNAMIC), DYNAMIC, fit_mask, blocks,
                                        MAX_PARALLEL_BLOCKS, params={"model_role": "BaseB_CatBoost"})
print("Run:", run.info.run_name, run.info.run_id)

Run 292_BaseB_CatBoost: 12 blocks, 6 in parallel with 5 CPUs each


[1/12] m05: RMSE 324.2, trees 5715, 304 min


[2/12] m01: RMSE 458.9, trees 6658, 366 min


[3/12] m03: RMSE 315.1, trees 9647, 522 min


[4/12] m06: RMSE 431.5, trees 10050, 536 min


...

[9/12] m11: RMSE 385.3, trees 10106, 1035 min


[10/12] m12: RMSE 361.9, trees 8806, 1060 min


[11/12] m10: RMSE 258.2, trees 11736, 1073 min


[12/12] m09: RMSE 439.0, trees 11975, 1073 min


Run: 292_BaseB_CatBoost c492e5c19f3a4e6cac86cf956a48d68a


In [7]:
month = pd.to_datetime(oof[TIME], utc=True).dt.month
error = (oof["prediction"] - oof["target"]) ** 2
table = pd.DataFrame({"rmse": error.groupby(month).mean() ** 0.5,
                      "rmse_le_1h": error[oof["target"] <= 3600].groupby(month[oof["target"] <= 3600]).mean() ** 0.5,
                      "trees": [metrics.get(f"best_iteration_m{m:02d}") for m in sorted(month.unique())]})
table.round(1)

,rmse,rmse_le_1h,trees
SCHED_TIME_UTC_mvt,,,
1,458.9,180.3,6658
2,459.2,178.1,10190
3,315.1,166.3,9647
4,294.1,169.6,12261
5,324.2,172.9,5715
6,431.5,179.2,10050
7,635.7,207.7,10720
8,407.8,192.4,8590
9,439.0,182.5,11975


## Final model

Trained on all allowed 2025 rows with the median number of trees of the blocks, saved in the same MLflow run under
`refit/`. `09_corrector` copies it to `model/final`, where the prediction notebooks (`10`, `11`) read it.

In [8]:
# Visibility is sometimes missing for whole stations in later files; the 2025 airport median fills such gaps at
# prediction time (only airports where 2025 itself has visibility for at least half of the flights).
vis = data.groupby("ADEP_mvt")["weather_visibility_km"]
fill_medians = {"weather_visibility_km": vis.median()[vis.apply(lambda s: s.isna().mean()) < 0.5].dropna().to_dict()}

In [9]:
final, meta = runner.refit(run, MODEL, data, FEATURES + list(DYNAMIC), DYNAMIC, fit_mask, metrics, blocks,
                           extra={"fill_medians": fill_medians})
print(f"Refit on {meta['fit_rows']:,} rows with {meta['trees']} trees -> {run.info.run_name}")

Refit on 2,075,561 rows with 10078 trees -> 292_BaseB_CatBoost
